# 02 ベースライン（段C）：素の検索で否定クエリを並べる

データ：`../data/examples_us.parquet`・`../data/products_us.parquet`

ルール：コードは自分で書く。分からない関数は調べる。答えを見る前に、まず予想を1行書く。

今夜のゴール：TF-IDF で query 7751（alarm clock without radio）の候補を並べ、radio 付きが上に来るかを見る。

## 結果まとめ（2026-10-29）
- query 7751「alarm clock without radio」：候補 **17件**（E 15・S 1・I 1）、タイトルに radio があるのは **5件**
- 上位10件の radio 付き：**TF-IDF 4／5**（1・2・3・10位）、**BM25 5／5**（1・2・4・5・7位）→ BM25 の方が radio 付きを上に集めた
- 原因：どちらも単語の一致で点を付けるので、without は消え、radio の一致で点が上がる（モデルの問題）
- ラベル：radio 付き5件のうち3件が **E**。見た目はただの時計で、ラジオは裏のボタンや説明の下の方 → 付けた人は見た目で判断（ラベルの問題）
- → ラベル（nDCG）だけでは測れない。タイトル・説明の radio（トラップ率）も別に見る
- ⚠️ 1クエリ・17件中の10件なので、まだ傾向。03 でクエリを20件に増やして確かめる


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. データを読んで merge する（01 と同じ）
- examples と products を `['product_id', 'product_locale']` で merge

In [2]:
expample_data = pd.read_parquet('../data/examples_us.parquet')
df = pd.DataFrame(expample_data)
product_data = pd.read_parquet('../data/products_us.parquet')
df_products = pd.DataFrame(product_data)

df_merged = pd.merge(df, df_products, on=['product_id', 'product_locale'], how='left')

## 2. query 7751 の候補だけにする
- `df_q` を作る（`.copy()` を忘れずに）
- `in_title` 列（タイトルに radio があるか）も付ける
- 何件ある？ E/S/C/I の内訳は？

In [3]:
df_q = df_merged[df_merged['query_id'] == 7751].copy()
df_q['in_title'] = df_q['product_title'].str.contains('radio', case=False, na=False)
df_q['in_desc'] = df_q['product_description'].str.contains('radio', case=False, na=False)

print(df_q.shape)

print(df_q['esci_label'].value_counts())
print(df_q['esci_label'].value_counts(normalize=True))
print()

(17, 16)
esci_label
E    15
S     1
I     1
Name: count, dtype: int64
esci_label
E    0.882353
S    0.058824
I    0.058824
Name: proportion, dtype: float64



## 3. 前処理：文書の列を作る
- 文書＝タイトル＋説明（description）を1列に
- HTML タグを消す（`str.replace(r'<[^>]+>', ' ', regex=True)`）、欠損は空文字に
- 予想：説明が空の商品はどれくらいある？

In [4]:
df_q['doc'] = df_q['product_title'].fillna('') + ' ' + df_q['product_description'].fillna('')
df_q['doc'].dtypes

df_q['doc'].isna().sum()

df_q['doc'] = df_q['doc'].str.replace(r'<[^>]+>', ' ', regex=True)
pd.set_option('display.max_colwidth', None)
df_q['doc']



69333                                                                                                                                                                                                                                                                                  RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom  Decent Alarm Clock with Large Clear Display    This clock radio equipped with a large clear red LED screen, the time can be easily read anywhere in the room, even without your glasses.  Decent look, perfect size, simple and beautiful design. It can even be used as an additional decoration. Suitable for living room, kitchen, office, bedroom, etc.     Thoughtful Dual-alarm Design   You can set two separate wake-up alarms for you and your partner on the same day.      Enjoy FM/ AM Radios   Turn on the radio, lean on the cou

## 4. モデル：TF-IDF でベクトルにする
- `TfidfVectorizer` を文書で `fit_transform`
- query は同じ vectorizer で `transform`（fit しない。なぜ？を1行で）
- 行列の形（`shape`）は？

In [5]:
df_q['doc'].shape
vectorize = TfidfVectorizer()
tfidf_matrix = vectorize.fit_transform(df_q['doc'])
tfidf_matrix

pd.DataFrame(tfidf_matrix.toarray(), columns=vectorize.get_feature_names_out()).iloc[:, :20]


,05,0mhz,0x3,10,100,1000ma,108,11in,12,120,122,12hrs,1412,15,16,18,180,20,2020,23
0,0.000000,0.000000,0.000000,0.043175,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.06052,0.000000,0.000000,0.105689,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.064293,0.032146,0.000000,0.000000,0.00000,0.000000,0.039346,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.072677,0.000000,0.000000,0.000000
6,0.052563,0.052563,0.000000,0.037499,0.000000,0.000000,0.052563,0.00000,0.037499,0.000000,0.000000,0.052563,0.000000,0.045897,0.00000,0.000000,0.000000,0.045897,0.000000,0.052563
7,0.000000,0.000000,0.000000,0.000000,0.040394,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.056621,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.103115,0.000000,0.000000,0.103115,0.000000,0.00000,0.073564,0.000000,0.103115,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.041249,0.000000,0.000000,0.05782,0.082498,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000


### なぜ否定が効かないのか（①モデルの問題）
- 検索文「alarm clock **without** radio」→ TF-IDF は単語に分けるだけ：alarm / clock / without / radio
- 「without が radio を打ち消す」という意味は分からない。**radio は「一致したら点が上がる単語」の1つ**
- radio がある時計 → alarm・clock・radio の3語が一致 → 点が高い → 上に来る／radio がない時計 → 2語だけ → 点が低い
- ＝「ラジオ**なし**」と調べたのに、ラジオ**付き**が上に来る。これが「否定が効かない」の正体
- 確かめ方：query を「alarm clock **with** radio」に変えても並びがほぼ同じ → 否定を無視している証拠

### 見るのは2つ
- ① モデル（検索）の問題：並べ直したら radio 付きが何位に来るか（上位5件に何件）
- ② データ（ラベル）の問題：ラジオなしで調べたのに、タイトル・説明に radio があって E になっている件数（in_title・in_desc）。タイトルでは15件中3件
- nDCG＝「良い物（E）が上に並んでいるか」0〜1。ただし radio 付きでも E なら良い物扱い → このクエリはトラップ率（①）も一緒に見る


## 5. 点数を付けて並べる
- `cosine_similarity` で query と各文書の点数 → `score` 列
- `score` で並べ、`esci_label`・`in_title`・`product_title` と一緒に見る
- 予想：上位5件に radio 付きは何件？

In [6]:
query = df_q['query'].iloc[0]
query_vector = vectorize.transform([query])

df_q['score'] = cosine_similarity(query_vector, tfidf_matrix)[0]
print(df_q['score'].describe())
print(df_q['score'].shape)
df_q['score']

count    17.000000
mean      0.136352
std       0.069442
min       0.040831
25%       0.111349
50%       0.122229
75%       0.144723
max       0.287493
Name: score, dtype: float64
(17,)


69333    0.272067
69334    0.112883
69335    0.113083
69336    0.040831
69337    0.196058
69338    0.115281
69339    0.287493
69340    0.140971
69341    0.060634
69342    0.124664
69343    0.111349
69344    0.122229
69345    0.209517
69346    0.062978
69347    0.144723
69348    0.072187
69349    0.131034
Name: score, dtype: float64

In [7]:
cols = ['score', 'esci_label', 'in_title', 'product_title']
result = df_q.sort_values(by='score', ascending=False)[cols]
result

,score,esci_label,in_title,product_title
69339,0.287493,E,True,"HAPTIME Digital Alarm Clock Radio: 6.2” Large LED Display with 4 Brightness Dimmer, Dual Alarms, Snooze, 12/24H, FM Radio with Sleep Timer, Blue Digits Clock for Home Bedside Bedroom"
69333,0.272067,E,True,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom"
69345,0.209517,I,True,"DreamSky Alarm Clock Radio for Bedroom - FM Radio Clock with Battery Backup, USB Charing Port, 1.2 Inch Bold Digit 0%-100% Dimmer, Adjustable Alarm Volume, Temperature, Snooze, Sleep Timer, 12/24H"
69337,0.196058,E,False,"Sharp Digital Alarm Clock – Black Case with Red LEDs - Ascending Alarm Grows Increasing Louder, Gentle Wake Up Experience, Dual Alarm - Battery Back-up, Easy to Use with Simple Operation"
69347,0.144723,E,False,RCA Digital Alarm Clock with Night Light
69340,0.140971,E,False,"Small LED Digital Alarm Clock with Nightlight, Easy to Set, Full Range Brightness Dimmer, HI-Low Volume, USB Charging Port, Snooze, Outlet Power Clock for Senior Bedrooms, Bedside, Desk"
69349,0.131034,E,False,"【2020 Upgraded】 Digital Wooden Alarm Clock, with 3 Alarm Settings, Electronic LED Time Display, 3 Level Brightness & Temperature, Good for Bedroom, Bedside, Desk, Office, Kids and Families, White"
69342,0.124664,E,False,"USCCE Small LED Digital Alarm Clock with Snooze, Easy to Set, Full Range Brightness Dimmer, Adjustable Alarm Volume with 5 Alarm Sounds, USB Charger, 12/24Hr, Compact Clock for Bedrooms, Bedside, Desk"
69344,0.122229,E,False,"REACHER Small LED Digital Alarm Clock with Snooze, Simple to Operate, Full Range Brightness Dimmer, Adjustable Alarm Volume, Outlet Powered Compact Clock for Bedrooms, Bedside, Desk, Shelf(White)"
69338,0.115281,S,True,"Antomove Projection Alarm Clock,6.7"" Curved-Screen Large Digital Display,Adjust Brightness Automatically,Dual Alarm Clock with 2 Alarm Sounds,FM Radio,Suitable for Night (Black/White)"


In [8]:
result.head(5)['in_title'].sum()

np.int64(3)

## 6. 気づき（自分の言葉で）
- 上位に radio 付きは来た？ それはなぜ？
- E の商品はどの辺にいた？
- 次（BM25・埋め込み・nDCG）で確かめたいこと：

### 2026-10-25 の気づき（Discord で整理）
- **fit_transform と transform の違い**：fit で商品17件から辞書（805語）と IDF を作る。transform はクエリの単語をその辞書で引いて数えるだけ。辞書に無い語は捨てる → クエリも (1, 805)
- **without は消える**：without は商品の文にほぼ出ないので辞書に無い → クエリから消える＝否定の情報が点数に届いていない
- **似ている＝良い、が否定で壊れる**：TF-IDF は「クエリに似ているほど上」。without radio だと radio が一致するほど点が上がり、ラジオ付きが上に来る。本当に欲しいのは clock には近く radio からは遠い商品 → これが ConstraintRAG の問題そのもの
- **数字の列**：初期設定だと 12・24h・50hz なども単語になる。ラジオの商品は周波数・時間の数字が多い（前処理で考える材料）
- **アイデア：クエリのリライト**：「alarm clock without radio」→ radio を消して「alarm clock」にすれば、radio で点が上がらなくなる。ただし radio 付きが下がるわけではない（同点になるだけ）→ 消す＋ radio を含む商品を下げる／外す（フィルター）まで要りそう。参考：`papers/negation_rewrite.pdf`


## 7. おまけ：with radio で並びは変わる？
- 上の 13〜15 のコードを、query だけ `alarm clock with radio` に変えて実行
- 並びがほぼ同じなら、TF-IDF は without を無視している証拠

In [9]:
query_with = 'alarm clock with radio'
query_vector_with = vectorize.transform([query_with])

df_q['score_with'] = cosine_similarity(query_vector_with, tfidf_matrix)[0]

cols_with = ['score', 'score_with', 'esci_label', 'in_title', 'product_title']
result_with = df_q.sort_values(by='score_with', ascending=False)[cols_with]
print('上位5件の radio 付き（with）:', result_with.head(5)['in_title'].sum())
print('without と with で順位が同じか:', list(result.index) == list(result_with.index))
print('without が辞書にあるか:', 'without' in vectorize.vocabulary_, '／ with:', 'with' in vectorize.vocabulary_)
result_with

上位5件の radio 付き（with）: 3
without と with で順位が同じか: False
without が辞書にあるか: True ／ with: True


,score,score_with,esci_label,in_title,product_title
69339,0.287493,0.402850,E,True,"HAPTIME Digital Alarm Clock Radio: 6.2” Large LED Display with 4 Brightness Dimmer, Dual Alarms, Snooze, 12/24H, FM Radio with Sleep Timer, Blue Digits Clock for Home Bedside Bedroom"
69333,0.272067,0.345037,E,True,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom"
69345,0.209517,0.288596,I,True,"DreamSky Alarm Clock Radio for Bedroom - FM Radio Clock with Battery Backup, USB Charing Port, 1.2 Inch Bold Digit 0%-100% Dimmer, Adjustable Alarm Volume, Temperature, Snooze, Sleep Timer, 12/24H"
69347,0.144723,0.266053,E,False,RCA Digital Alarm Clock with Night Light
69337,0.196058,0.226179,E,False,"Sharp Digital Alarm Clock – Black Case with Red LEDs - Ascending Alarm Grows Increasing Louder, Gentle Wake Up Experience, Dual Alarm - Battery Back-up, Easy to Use with Simple Operation"
69343,0.111349,0.221757,E,False,"Sharp LED Digital Alarm Clock – Simple Operation - Easy to See Large Numbers, Built in Night Light, Loud Beep Alarm with Snooze, Bright Big Red Digit Display"
69342,0.124664,0.215287,E,False,"USCCE Small LED Digital Alarm Clock with Snooze, Easy to Set, Full Range Brightness Dimmer, Adjustable Alarm Volume with 5 Alarm Sounds, USB Charger, 12/24Hr, Compact Clock for Bedrooms, Bedside, Desk"
69340,0.140971,0.190423,E,False,"Small LED Digital Alarm Clock with Nightlight, Easy to Set, Full Range Brightness Dimmer, HI-Low Volume, USB Charging Port, Snooze, Outlet Power Clock for Senior Bedrooms, Bedside, Desk"
69349,0.131034,0.187356,E,False,"【2020 Upgraded】 Digital Wooden Alarm Clock, with 3 Alarm Settings, Electronic LED Time Display, 3 Level Brightness & Temperature, Good for Bedroom, Bedside, Desk, Office, Kids and Families, White"
69344,0.122229,0.187251,E,False,"REACHER Small LED Digital Alarm Clock with Snooze, Simple to Operate, Full Range Brightness Dimmer, Adjustable Alarm Volume, Outlet Powered Compact Clock for Bedrooms, Bedside, Desk, Shelf(White)"


## 8. 確認：上に来たのは文の長さのせい？（2026-10-25）
- radio を入れないクエリ「alarm clock」でも並べて比べる
- radio 付きの方が説明が長い（＝薄まって点が下がる側）なのに上に来るなら、原因は長さではなく「radio」の一致

In [10]:
df_q['len'] = df_q['doc'].str.split().str.len()
for qtext in ['alarm clock without radio', 'alarm clock']:
    s = cosine_similarity(vectorize.transform([qtext]), tfidf_matrix)[0]
    top5 = df_q.assign(s=s).sort_values('s', ascending=False).head(5)
    print(qtext, '| 上位5件の radio 付き:', top5['in_title'].sum(), '| 単語数:', top5['len'].tolist())
print('単語数の中央値 radio 付き / それ以外:', df_q[df_q['in_title']]['len'].median(), '/', df_q[~df_q['in_title']]['len'].median())

alarm clock without radio | 上位5件の radio 付き: 3 | 単語数: [338, 295, 309, 228, 7]
alarm clock | 上位5件の radio 付き: 0 | 単語数: [7, 228, 50, 326, 27]
単語数の中央値 radio 付き / それ以外: 309.0 / 116.0


### 結果
- without radio：上位5件中 radio 付き3件／alarm clock：0件
- 単語数の中央値：radio 付き 309、それ以外 116
- → 長さではなく、クエリの「radio」の一致で上に来ている。結論は言い過ぎではない

In [37]:
from rank_bm25 import BM25Okapi
import re

tokenized_docs = [doc.lower().split() for doc in df_q['doc']]
print(len(tokenized_docs))
print(tokenized_docs[0][:10])

tokenized_docs = [re.findall(r'\b\w\w+\b', doc.lower()) for doc in df_q['doc']]
print(tokenized_docs[0][:10])

17
['runningbyte', 'digital', 'alarm', 'clock', 'radio-', 'am/fm', 'radio', 'with', 'preset,', 'large']
['runningbyte', 'digital', 'alarm', 'clock', 'radio', 'am', 'fm', 'radio', 'with', 'preset']


In [38]:
bm25 = BM25Okapi(tokenized_docs)
print(bm25.avgdl)
print(bm25.idf['radio'], bm25.idf['clock'])

179.2941176470588
0.5705448584676129 0.4770513468463255


In [44]:
docs = ["alarm clock radio", "alarm clock without radio", "radio speaker"]
index = {}

for i, doc in enumerate(docs, start=1):
    print(f"Document {i}: {doc}")
    
    for word in doc.split():
        if word not in index:
            index[word] = []
        index[word].append(i)

print("Inverted Index:", index)

print(index["radio"])

def intersect(a,b):
    i, j = 0,0
    result = []
    
    while i < len(a) and j < len(b):
        if a[i] == b[j]:    
            result.append(a[i])
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1  
    return result
print(intersect(index["alarm"], index["radio"]))  # → [1, 2]
    
    
    

Document 1: alarm clock radio
Document 2: alarm clock without radio
Document 3: radio speaker
Inverted Index: {'alarm': [1, 2], 'clock': [1, 2], 'radio': [1, 2, 3], 'without': [2], 'speaker': [3]}
[1, 2, 3]
[1, 2]


In [45]:
q = re.findall(r'\b\w\w+\b', 'alarm clock without radio'.lower())
s = bm25.get_scores(q)
top5 = df_q.assign(s=s).sort_values('s', ascending=False).head(5)
print(top5['in_title'].sum())

4


## 10. Step 4：TF-IDF と BM25 を上位10件で比べる（今日）

⚠️ 候補は全部で何件？（17件なら、上位10件＝半分以上。差が出にくいことを先に頭に入れておく）

**第1ステップ**：まず予想を1行 →「BM25 の上位10件に radio 付きは何件？ TF-IDF より多い？」
1. TF-IDF の上位10件で `in_title` を合計（セル 15 の `head(5)` を `head(10)` に）
2. BM25 の上位10件で同じく合計（セル 26 の `head(5)` を `head(10)` に）
3. 2つを1行に並べて print（例：`TF-IDF: 3 / BM25: 4`）
4. 気づきを1〜2行：差は何件？ 1件差なら「ばらつきかも」と書く

**次（今日やれたら）**
- クエリを `alarm clock` にして同じ比較 → radio を抜くと差は消える？
- 単語ごとの内訳：`bm25.get_scores(['radio'])` のように1語ずつ点を出して並べる（やり方は作業中に相談）


In [48]:
result.head(10)['in_title'].sum()
result.head(10)

,score,esci_label,in_title,product_title
69339,0.287493,E,True,"HAPTIME Digital Alarm Clock Radio: 6.2” Large LED Display with 4 Brightness Dimmer, Dual Alarms, Snooze, 12/24H, FM Radio with Sleep Timer, Blue Digits Clock for Home Bedside Bedroom"
69333,0.272067,E,True,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom"
69345,0.209517,I,True,"DreamSky Alarm Clock Radio for Bedroom - FM Radio Clock with Battery Backup, USB Charing Port, 1.2 Inch Bold Digit 0%-100% Dimmer, Adjustable Alarm Volume, Temperature, Snooze, Sleep Timer, 12/24H"
69337,0.196058,E,False,"Sharp Digital Alarm Clock – Black Case with Red LEDs - Ascending Alarm Grows Increasing Louder, Gentle Wake Up Experience, Dual Alarm - Battery Back-up, Easy to Use with Simple Operation"
69347,0.144723,E,False,RCA Digital Alarm Clock with Night Light
69340,0.140971,E,False,"Small LED Digital Alarm Clock with Nightlight, Easy to Set, Full Range Brightness Dimmer, HI-Low Volume, USB Charging Port, Snooze, Outlet Power Clock for Senior Bedrooms, Bedside, Desk"
69349,0.131034,E,False,"【2020 Upgraded】 Digital Wooden Alarm Clock, with 3 Alarm Settings, Electronic LED Time Display, 3 Level Brightness & Temperature, Good for Bedroom, Bedside, Desk, Office, Kids and Families, White"
69342,0.124664,E,False,"USCCE Small LED Digital Alarm Clock with Snooze, Easy to Set, Full Range Brightness Dimmer, Adjustable Alarm Volume with 5 Alarm Sounds, USB Charger, 12/24Hr, Compact Clock for Bedrooms, Bedside, Desk"
69344,0.122229,E,False,"REACHER Small LED Digital Alarm Clock with Snooze, Simple to Operate, Full Range Brightness Dimmer, Adjustable Alarm Volume, Outlet Powered Compact Clock for Bedrooms, Bedside, Desk, Shelf(White)"
69338,0.115281,S,True,"Antomove Projection Alarm Clock,6.7"" Curved-Screen Large Digital Display,Adjust Brightness Automatically,Dual Alarm Clock with 2 Alarm Sounds,FM Radio,Suitable for Night (Black/White)"


In [53]:
q = re.findall(r'\b\w\w+\b', 'alarm clock without radio'.lower())
s = bm25.get_scores(q)
top10 = df_q.assign(s=s).sort_values('s', ascending=False).head(10)
print(top10['in_title'].sum())

df_q['in_title'].sum()

5


np.int64(5)

In [54]:
top10[['s', 'esci_label', 'in_title', 'product_title']]


,s,esci_label,in_title,product_title
69333,3.562666,E,True,"RunningByte Digital Alarm Clock Radio- AM/FM Radio with preset, Large LED Display with Dimmer, Dual Alarm, Adjustable Alarm Volume, Snooze, Sleep Timer, Easy to Operate Clock Radios for Bedroom"
69345,3.299622,I,True,"DreamSky Alarm Clock Radio for Bedroom - FM Radio Clock with Battery Backup, USB Charing Port, 1.2 Inch Bold Digit 0%-100% Dimmer, Adjustable Alarm Volume, Temperature, Snooze, Sleep Timer, 12/24H"
69337,3.090461,E,False,"Sharp Digital Alarm Clock – Black Case with Red LEDs - Ascending Alarm Grows Increasing Louder, Gentle Wake Up Experience, Dual Alarm - Battery Back-up, Easy to Use with Simple Operation"
69338,3.090054,S,True,"Antomove Projection Alarm Clock,6.7"" Curved-Screen Large Digital Display,Adjust Brightness Automatically,Dual Alarm Clock with 2 Alarm Sounds,FM Radio,Suitable for Night (Black/White)"
69339,3.029057,E,True,"HAPTIME Digital Alarm Clock Radio: 6.2” Large LED Display with 4 Brightness Dimmer, Dual Alarms, Snooze, 12/24H, FM Radio with Sleep Timer, Blue Digits Clock for Home Bedside Bedroom"
69340,2.524184,E,False,"Small LED Digital Alarm Clock with Nightlight, Easy to Set, Full Range Brightness Dimmer, HI-Low Volume, USB Charging Port, Snooze, Outlet Power Clock for Senior Bedrooms, Bedside, Desk"
69334,2.494173,E,True,"Sunrise Alarm Clock, Wake Up Light with Sunrise/Sunset Simulation, Dual Alarms with FM Radio, 7 Nature Sounds & Snooze, 7 Colors Night Light, Bedroom Digital Alarm Clock for Heavy Sleepers Adults Kids"
69335,2.001289,E,False,"SHARP Digital Alarm Clock with Keyboard Style Controls, Battery Back-up, Easy to Use with Simple Operation, Black Case with Red LED Display"
69349,1.939572,E,False,"【2020 Upgraded】 Digital Wooden Alarm Clock, with 3 Alarm Settings, Electronic LED Time Display, 3 Level Brightness & Temperature, Good for Bedroom, Bedside, Desk, Office, Kids and Families, White"
69343,1.929043,E,False,"Sharp LED Digital Alarm Clock – Simple Operation - Easy to See Large Numbers, Built in Night Light, Loud Beep Alarm with Snooze, Bright Big Red Digit Display"
